In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

### UC setup

In [0]:
%sql
USE CATALOG learn_adb_fikrat;
create schema if not exists bronze;
create schema if not exists silver;
Use bronze;

In [0]:
%sql
drop table if exists audit;
-- drop table if exists iot_measurements;
-- drop table if exists silver.iot_measurements_aggregated


In [0]:
checkpoint_root_path = "/Volumes/learn_adb_fikrat/bronze/ext_landing_volume/streaming-checkpoints/eventhub"
checkpoint_path_sensor=f'{checkpoint_root_path}/iot_measurements'
checkpoint_path_sensor1=f'{checkpoint_root_path}/iot_measurements1'
checkpoint_path_sensor2=f'{checkpoint_root_path}/iot_measurements2'
checkpoint_path_sensor3=f'{checkpoint_root_path}/iot_measurements3'

In [0]:
dbutils.fs.rm(checkpoint_path_sensor, True)
dbutils.fs.rm(checkpoint_path_sensor1, True)
dbutils.fs.rm(checkpoint_path_sensor2, True)
dbutils.fs.rm(checkpoint_path_sensor3, True)

### Event Hub configurations

In [0]:
connection_string_ehs = dbutils.secrets.get(scope = "fikrats_study_scope", key = "eh-dbr-source-connstr")

In [0]:
event_hub_namespace="eh-dbr"
source_event_hub_name="dbr-source"

### Reading from Event Hub

In [0]:
ehConf={}
ehConf['eventhubs.connectionString'] = sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connection_string_ehs)
ehConf['eventhubs.eventHubName']=source_event_hub_name


In [0]:
df=spark.readStream.format("eventhubs").options(**ehConf).load()
# display(df)

In [0]:
df1=df.withColumn("event_payload", F.col("body").cast("string"))\
    .select("partition","offset","enqueuedTime","event_payload")
# display(df1)    


In [0]:
json_schema=StructType([
    StructField("records", ArrayType(
        StructType([
            StructField("resourceId", StringType(), True),
            StructField("operationVersion", StringType(), True),
            StructField("identity", StringType(), True),
            StructField("operationName", StringType(), True),
            StructField("time", StringType(), True),
            StructField("category", StringType(), True),
            StructField("properties", StructType([
                StructField("sourceIPAddress", NullType(), True),
                StructField("logId", StringType(), True),
                StructField("serviceName", StringType(), True),
                StructField("userAgent", StringType(), True),
                StructField("response", NullType(), True),
                StructField("sessionId", StringType(), True),
                StructField("actionName", StringType(), True),
                StructField("requestId", StringType(), True),
                StructField("requestParams", StringType(), True)
            ]), True),
            StructField("Host", StringType(), True)
        ])
    ), True)
])

In [0]:
df2=df1.withColumn("records_json", F.from_json("event_payload", json_schema))\
    .withColumn("records",F.explode("records_json.records"))\
    .select('records.*')\
    .select('*','properties.*')\
    .drop('event_payload','records_json','records','properties')
    # .filter ("serviceName=='databrickssql'")
display(df2)    

In [0]:
stm_brz = df2.drop('sourceIPAddress','response','sessionID','operationVersion')\
        .writeStream.format('delta')\
        .outputMode('append')\
        .option("checkpointLocation", checkpoint_path_sensor2)\
        .toTable('audit')

In [0]:
%sql
select * from audit where serviceName='databrickssql' 
and actionName in ('commandFinish','commandSubmit')
--and requestParams like '%iot_measure%'
--where actionName='commandSubmit' --and requestParams like '%weather%' 

In [0]:
json_schema = StructType([
    StructField("warehouseId", StringType(), True),
    StructField("commandId", StringType(), True),
    StructField("commandText", StringType(), True)
])
id_schema = StructType([
    StructField("email", StringType(), True),
    StructField("subjectName", StringType(), True)
])

df=spark.table('audit').withColumn("requestParams", F.from_json("requestParams", json_schema))\
    .withColumn("identity", F.from_json("identity", id_schema))\
    .selectExpr("identity.*","cast(time as timestamp) as time","requestId", "serviceName","actionName", "requestParams.*")\
    .where("serviceName = 'databrickssql' and actionName in ('commandSubmit','commandFinish') and email like '%slalom.com'")\
    .orderBy("requestId")
display(df)
df.createOrReplaceTempView("tvw_commandSubmit")

In [0]:
# json_schema = StructType([
#     StructField("warehouseId", StringType(), True),
#     StructField("commandId", StringType(), True)
# ])
# df=spark.table('audit').withColumn("requestParams", F.from_json("requestParams", json_schema))\
#     .selectExpr("cast(time as timestamp) as time","serviceName","actionName", "requestParams.*")\
#     .where("serviceName = 'databrickssql' and actionName in ('commandFinish')")
# display(df)
# df.createOrReplaceTempView("tvw_commandFinish")

In [0]:
%sql
select * from tvw_commandSubmit

In [0]:
%sql
WITH CTE AS (
SELECT requestId,min(time),max(time),max(commandText), 
    TIMESTAMPDIFF(MILLISECOND, MIN(time), MAX(time)) AS duration_milli_seconds 
    FROM  tvw_commandSubmit
GROUP BY requestId)
SELECT * FROM CTE --where  duration_milli_seconds>0

## Using Audit system table

In [0]:
%sql
SELECT * FROM 
    system.access.audit
WHERE 
    action_name IN ('commandSubmit', 'commandFinish')
    and user_identity.email != 'System-User' and user_identity.email like '%slalom.com'
    and identity_metadata.run_by like '%slalom.com'

In [0]:
%sql
WITH CTE AS (
SELECT request_id,min(event_time),max(event_time),max(request_params['commandText']), 
    TIMESTAMPDIFF(MILLISECOND, MIN(event_time), MAX(event_time)) AS duration_milli_seconds FROM 
    system.access.audit
WHERE 
    action_name IN ('commandSubmit', 'commandFinish') 
    -- AND user_identity.email != 'System-User' AND user_identity.email LIKE '%slalom.com' 
    and identity_metadata.run_by like '%slalom.com'
GROUP BY request_id)
  SELECT * FROM CTE where  duration_milli_seconds>0
    

In [0]:
%sql
SELECT 
    commandid,
    MIN(event_time) AS submit_time,
    MAX(event_time) AS finish_time,
    TIMESTAMPDIFF(SECOND, MIN(event_time), MAX(event_time)) AS duration_seconds
FROM 
    system.access.audit
WHERE 
    action_name IN ('commandSubmit', 'commandFinish')
GROUP BY 
    commandid
ORDER BY 
    duration_seconds DESC;

In [0]:
%sql
select actionName, from_json(requestParams,'warehouseid STRING, command STRING') as requestParams
from audit 
where serviceName = 'databrickssql' 
-- and actionName in ('commandSubmit', 'commandFinish')
and actionName in ('commandFinish')

In [0]:
for strm in spark.streams.active:
    # print(strm.name)
    strm.stop()